# TP 2 — Clustering avec k-means

**Cours :** *Clustering with k-Means* (slides 41–63) — variance intra-cluster, partitions de Voronoï centroïdales, cas exacts ($k=1$, $d=1$), heuristique de Lloyd, initialisations (aléatoires, k-means++), choix de $k$ (coude, silhouette).

**À la fin de ce notebook, vous saurez :**
1. coder l'algorithme de **Lloyd** et vérifier qu'il fait décroître la variance à chaque étape ;
2. coder et comparer les **initialisations** (boîte englobante, Forgy, partition aléatoire, **k-means++**) ;
3. résoudre **exactement** le k-means en dimension 1 par **programmation dynamique** — et l'utiliser pour segmenter la volatilité en régimes ;
4. choisir $k$ avec le **coude** et la **silhouette** (codée à la main) ;
5. regrouper des actions par leur comportement et confronter le résultat aux secteurs GICS.

> 📝 **Version énoncé.** Complétez les blocs `# TODO`, puis exécutez la cellule *Test* qui suit chaque exercice. Le corrigé est dans `../corriges/`.

In [ ]:
import sys, pathlib, warnings
_here = pathlib.Path.cwd()
ROOT = _here.parent if _here.name in ("exercices", "corriges") else _here
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tp_utils import *          # style graphique, check(), check_close(), timer()...
import tp_data                  # données réelles (cache / téléchargement) ou simulées

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
rng = np.random.default_rng(42)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, adjusted_rand_score
from sklearn.datasets import make_blobs

## Partie 1 — L'objectif : la variance intra-cluster

$$\min_{c_1,\dots,c_k,\ \sigma}\ \mathrm{Var}(P, c, \sigma) = \frac1n\sum_{p\in P}\|p - c_{\sigma(p)}\|_2^2 = \sum_{i=1}^k \frac{n_i}{n}\,\mathrm{Var}(C_i, c_i)$$

Deux propriétés du cours donnent tout l'algorithme :
- **partition fixée** → le meilleur centre de chaque cluster est son **centroïde** (car $\mathrm{Var}(C, c) = \mathrm{Var}(C, c^*) + \|c^* - c\|^2$) ;
- **centres fixés** → la meilleure partition est celle de **Voronoï** (chaque point va au centre le plus proche).

### Exercice 1.1 — Briques de base
- `pairwise_sq_dists(X, C)` : matrice $n\times k$ des $\|x_i - c_j\|^2$, **sans boucle** (développez $\|x\|^2 - 2x\cdot c + \|c\|^2$).
- `assign(X, C)` : étiquettes de Voronoï.
- `update_centers(X, labels, C_old)` : centroïdes ; un cluster vide garde son ancien centre.
- `kmeans_cost(X, C, labels)` : la variance ci-dessus.

In [ ]:
def pairwise_sq_dists(X, C):
    # TODO : développer ||x||² − 2 x·c + ||c||² avec des produits matriciels ; tronquer à 0 les petites valeurs négatives
    raise NotImplementedError("À compléter")


def assign(X, C):
    # TODO : argmin des distances² sur l'axe des centres
    raise NotImplementedError("À compléter")


def update_centers(X, labels, C_old):
    # TODO : np.bincount ou boucle sur les k clusters
    raise NotImplementedError("À compléter")


def kmeans_cost(X, C, labels):
    # TODO : moyenne des ||x − c_label(x)||² (C[labels] donne le centre de chaque point)
    raise NotImplementedError("À compléter")

In [ ]:
# Test
X = rng.normal(size=(500, 3)); C = rng.normal(size=(4, 3))
D = pairwise_sq_dists(X, C)
check_close(D, ((X[:, None] - C[None]) ** 2).sum(-1), "pairwise_sq_dists", rtol=1e-6, atol=1e-9)
lab = assign(X, C)
C2 = update_centers(X, lab, C)
check_close(C2[0], X[lab == 0].mean(0), "update_centers = centroïdes")
check(kmeans_cost(X, C2, lab) <= kmeans_cost(X, C, lab) + 1e-12, "le centroïde fait baisser la variance")
check(kmeans_cost(X, C2, assign(X, C2)) <= kmeans_cost(X, C2, lab) + 1e-12, "Voronoï fait baisser la variance")

## Partie 2 — L'heuristique de Lloyd

Trouver le minimum global est NP-difficile dès que $k>1$ et $d>1$. Lloyd alterne les deux minimisations partielles :
```
initialiser c_1..c_k
répéter : σ ← Voronoï(c) ;  c_i ← centroïde(C_i)    jusqu'à ce que σ ne change plus
```
La variance **décroît strictement** à chaque itération non terminale et le nombre de partitions de Voronoï est fini → **convergence en un nombre fini d'itérations**, vers un minimum **local**.

### Exercice 2.1 — `lloyd`

In [ ]:
def lloyd(X, C0, max_iter=300):
    """Retourne (centres, étiquettes, historique des coûts)."""
    C = C0.astype(float).copy()
    labels = assign(X, C)
    history = [kmeans_cost(X, C, labels)]
    for _ in range(max_iter):
        # TODO : mise à jour des centres, puis des étiquettes ; arrêt si étiquettes inchangées
        raise NotImplementedError("À compléter")
    return C, labels, history

In [ ]:
# Test : même résultat que scikit-learn à initialisation égale, coût décroissant
X, y_true = make_blobs(n_samples=1500, centers=6, cluster_std=1.3, random_state=3)
C0 = X[rng.choice(len(X), 6, replace=False)]
C, lab, hist = lloyd(X, C0)
sk = KMeans(6, init=C0, n_init=1, algorithm="lloyd", tol=0, max_iter=300).fit(X)
check(np.all(np.diff(hist) <= 1e-12), "la variance ne remonte jamais")
check_close(hist[-1] * len(X), sk.inertia_, "même inertie que sklearn.KMeans", rtol=1e-8)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist, marker="o"); axes[0].set_xlabel("itération"); axes[0].set_title("Variance intra-cluster (Lloyd)")
axes[1].scatter(*X.T, c=[PALETTE[l] for l in lab], s=6)
axes[1].scatter(*C.T, marker="X", s=120, color=INK, label="centres"); axes[1].legend()
axes[1].set_title("Partition de Voronoï centroïdale obtenue")
plt.tight_layout(); plt.show()

## Partie 3 — Initialisation

Lloyd converge vers un minimum local qui **dépend de l'initialisation**. Le cours en présente quatre :
- `init_bbox` : centres uniformes dans la boîte englobante ;
- `init_forgy` : $k$ points de données tirés au hasard (Forgy 1965) ;
- `init_random_partition` : centroïdes d'une partition aléatoire (chaque point reçoit une étiquette uniforme) ;
- `init_kmeanspp` : **k-means++** — $c_1$ uniforme, puis $c_i$ tiré avec probabilité $\propto d(p, \{c_1..c_{i-1}\})^2$. Garantie : $\mathbb{E}[\mathrm{Var}] \le 8(2+\ln k)\,\mathrm{OPT}$.

### Exercice 3.1

In [ ]:
def init_bbox(X, k, rng):
    # TODO : rng.uniform entre X.min(0) et X.max(0), taille (k, d)
    raise NotImplementedError("À compléter")


def init_forgy(X, k, rng):
    # TODO : k indices distincts : rng.choice(n, k, replace=False)
    raise NotImplementedError("À compléter")


def init_random_partition(X, k, rng):
    # TODO : étiquettes rng.integers(0, k, n), puis centroïde de chaque groupe
    raise NotImplementedError("À compléter")


def init_kmeanspp(X, k, rng):
    # TODO : maintenir d2 = distance² au centre le plus proche, tirer selon d2 / d2.sum()
    raise NotImplementedError("À compléter")

In [ ]:
# Test
X, _ = make_blobs(n_samples=600, centers=5, random_state=0)
for f in (init_bbox, init_forgy, init_random_partition, init_kmeanspp):
    C0 = f(X, 5, np.random.default_rng(0))
    check(C0.shape == (5, 2) and np.isfinite(C0).all(), f"{f.__name__} renvoie 5 centres finis")
# k-means++ choisit (presque toujours) des centres dans des blobs distincts
Xs, ys = make_blobs(n_samples=500, centers=5, cluster_std=0.3, random_state=1)
lab_true_of_center = lambda C: {ys[np.argmin(((Xs - c) ** 2).sum(1))] for c in C}
cover = np.mean([len(lab_true_of_center(init_kmeanspp(Xs, 5, np.random.default_rng(s)))) == 5 for s in range(100)])
check(cover > 0.8, f"k-means++ touche les 5 blobs dans {cover:.0%} des tirages")

### Expérience 3.2 — Taux de succès des initialisations

Données : 15 blobs bien séparés ($k=15$). On lance 100 fois chaque initialisation suivie de Lloyd, et on compte un **succès** quand le coût final est à moins de 1 % du meilleur coût observé (≈ optimum global).

In [ ]:
X15, _ = make_blobs(n_samples=3000, centers=15, cluster_std=0.6, center_box=(-20, 20), random_state=7)
inits = {"boîte englobante": init_bbox, "Forgy": init_forgy,
         "partition aléatoire": init_random_partition, "k-means++": init_kmeanspp}
final = {name: [] for name in inits}
for s in range(100):
    r = np.random.default_rng(1000 + s)
    for name, f in inits.items():
        final[name].append(lloyd(X15, f(X15, 15, r))[2][-1])
best = min(min(v) for v in final.values())
res = pd.DataFrame({name: {"taux de succès": np.mean(np.array(v) <= 1.01 * best),
                           "coût médian / meilleur": np.median(v) / best}
                    for name, v in final.items()}).T

fig, ax = plt.subplots(figsize=(8, 3.8))
for i, n in enumerate(inits):
    v = np.array(final[n]) / best
    ax.scatter(v, i + rng.uniform(-0.15, 0.15, len(v)), s=10, color=PALETTE[i], alpha=0.6)
    ax.plot([np.median(v)] * 2, [i - 0.3, i + 0.3], color=INK, lw=2)
ax.set_yticks(range(len(inits)), list(inits)); ax.set_xscale("log")
ax.set_xlabel("coût final / meilleur coût (log) — trait noir : médiane")
ax.set_title("Lloyd : qualité du minimum local selon l'initialisation (100 runs)")
plt.show()
res

**À retenir :** les initialisations uniformes se dégradent vite quand $k$ ou $d$ augmente ; **k-means++ est le choix par défaut** (c'est celui de scikit-learn), avec plusieurs redémarrages (`n_init`) en gardant le meilleur coût.

## Partie 4 — Cas exacts : $k = 1$ et $d = 1$

- $k = 1$ : la solution est le centroïde, $O(nd)$.
- $d = 1$ : les cellules de Voronoï sont des intervalles, donc les clusters optimaux sont **contigus** une fois les points triés. D'où la récurrence (slide 53), avec $\mathrm{SSE}(j..i) = \sum_{l=j}^{i}(p_l - \bar p_{j..i})^2$ :
$$\mathrm{OPT}(i, k) = \min_{1\le j\le i}\ \big\{\mathrm{OPT}(j-1, k-1) + \mathrm{SSE}(j..i)\big\},\qquad \mathrm{OPT}(0,0)=0.$$

Avec les **sommes cumulées** $S_1(i) = \sum_{l\le i} p_l$ et $S_2(i) = \sum_{l\le i} p_l^2$, on a $\mathrm{SSE}(j..i) = (S_2(i) - S_2(j-1)) - (S_1(i)-S_1(j-1))^2 / (i-j+1)$ en $O(1)$, donc l'algorithme est en $O(kn^2)$ — en vectorisant la minimisation sur $j$.

### Exercice 4.1 — `kmeans_1d_dp`
Retourne le coût optimal (variance, donc SSE / n) et les étiquettes **dans l'ordre d'origine** des points (cluster 0 = valeurs les plus basses).

In [ ]:
def kmeans_1d_dp(x, k):
    x = np.asarray(x, float)
    order = np.argsort(x); p = x[order]; n = len(p)
    S1 = np.concatenate([[0.0], np.cumsum(p)])
    S2 = np.concatenate([[0.0], np.cumsum(p ** 2)])

    def sse(j, i):        # SSE des points p[j-1 .. i-1] (indices 1-based j..i), j peut être un tableau
        m = i - j + 1
        return (S2[i] - S2[j - 1]) - (S1[i] - S1[j - 1]) ** 2 / m

    OPT = np.full((n + 1, k + 1), np.inf); OPT[0, 0] = 0.0
    ARG = np.zeros((n + 1, k + 1), dtype=int)     # ARG[i, c] = début j du dernier cluster
    # TODO : double boucle (c, i), minimisation vectorisée sur j ; puis remontée des ARG
    raise NotImplementedError("À compléter")
    labels = np.empty(n, dtype=int); labels[order] = labels_sorted
    return OPT[n, k] / n, labels

In [ ]:
# Test : comparaison avec l'énumération exhaustive des coupures (petit n)
from itertools import combinations
def brute_force_1d(x, k):
    p = np.sort(x); n = len(p); best = np.inf
    for cuts in combinations(range(1, n), k - 1):
        b = (0,) + cuts + (n,)
        best = min(best, sum(((p[b[t]:b[t+1]] - p[b[t]:b[t+1]].mean()) ** 2).sum() for t in range(k)))
    return best / n

for trial in range(5):
    x = rng.normal(size=14) * rng.choice([1, 5], 14)
    cost, lab = kmeans_1d_dp(x, 3)
    check(np.isclose(cost, brute_force_1d(x, 3)), f"DP = force brute (essai {trial})")
check(np.isclose(cost, kmeans_cost(x[:, None], np.array([[x[lab == c].mean()] for c in range(3)]), lab)),
      "étiquettes cohérentes avec le coût")

### Application 4.2 — Régimes de volatilité du S&P 500

Une tâche classique : découper la volatilité réalisée en **régimes** (calme / normal / stress) pour conditionner une stratégie, un modèle de risque ou un sizing. C'est un clustering **en dimension 1** : la DP donne l'**optimum global**, alors que Lloyd peut rester bloqué. (En finance/cartographie on appelle aussi cette méthode *Jenks natural breaks*.)

On travaille sur le **log** de la volatilité réalisée à 21 jours (annualisée) — l'échelle log est plus naturelle pour une variable positive et asymétrique.

In [ ]:
spy, src = tp_data.load_index_returns("SPY", start="2000-01-01")
vol = spy.rolling(21).std().dropna() * np.sqrt(252)
logvol = np.log(vol)
print("source :", src, "| n =", len(logvol))

with timer("DP exacte (k = 3)"):
    cost_dp, reg = kmeans_1d_dp(logvol.to_numpy(), 3)

# Lloyd avec initialisation Forgy, 200 fois
costs_lloyd = []
for s in range(200):
    r = np.random.default_rng(s)
    x = logvol.to_numpy()[:, None]
    costs_lloyd.append(lloyd(x, init_forgy(x, 3, r))[2][-1])
costs_lloyd = np.array(costs_lloyd)
check(np.all(costs_lloyd >= cost_dp - 1e-12), "aucun run de Lloyd ne bat la DP (optimum global)")
print(f"Lloyd (Forgy) : {np.mean(costs_lloyd > cost_dp * (1 + 1e-9)):.0%} des runs sont sous-optimaux, "
      f"écart max = +{100 * (costs_lloyd.max() / cost_dp - 1):.3f} %")

In [ ]:
names = ["calme", "normal", "stress"]
fig, ax = plt.subplots(figsize=(11, 4))
for c in range(3):
    m = reg == c
    ax.scatter(vol.index[m], vol[m], s=3, color=[AQUA, BLUE, RED][c], label=names[c])
from matplotlib.ticker import PercentFormatter, FixedLocator
ax.set_yscale("log"); ax.set_ylabel("vol réalisée 21 j (annualisée, échelle log)")
ax.yaxis.set_major_locator(FixedLocator([0.05, 0.1, 0.2, 0.4, 0.8]))
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.yaxis.set_minor_locator(FixedLocator([]))
ax.set_title("Régimes de volatilité : k-means 1D exact (programmation dynamique)")
ax.legend(markerscale=4, loc="upper left"); plt.show()
pd.DataFrame({"part du temps": pd.Series(reg).value_counts(normalize=True).sort_index().to_numpy(),
              "vol min": [vol[reg == c].min() for c in range(3)],
              "vol max": [vol[reg == c].max() for c in range(3)]}, index=names)

**Lecture.** En dimension 1 avec peu de clusters, Lloyd tombe souvent tout près de l'optimum — mais *pas toujours*, et rien ne permet de le savoir sans la DP. La DP apporte la **garantie** d'optimalité et la **reproductibilité** (aucun aléa), pour un coût négligeable ici.

**Remarque pratique.** Ici les régimes sont calculés sur tout l'échantillon (analyse descriptive). Pour un usage en trading, les seuils devraient être estimés uniquement sur le passé (fenêtre glissante ou expansive), sinon on introduit un biais d'anticipation.

## Partie 5 — Choisir $k$

- **Méthode du coude** : tracer la variance optimale (approchée) en fonction de $k$ et repérer la cassure. Elle décroît toujours avec $k$ — on cherche le point où le gain devient marginal.
- **Silhouette** (Rousseeuw 1987) : pour chaque point, $a(p)$ = distance moyenne aux autres points de son cluster, $b(p)$ = plus petite distance moyenne à un autre cluster,
$$s(p) = \frac{b(p) - a(p)}{\max\{a(p), b(p)\}} \in [-1, 1].$$
On choisit le $k$ qui **maximise** la silhouette moyenne.

### Exercice 5.1 — Silhouette à la main
Convention : un point seul dans son cluster a $s(p) = 0$.

In [ ]:
def silhouette(X, labels):
    """Silhouettes individuelles s(p) (tableau de taille n)."""
    D = np.sqrt(pairwise_sq_dists(X, X))
    ks = np.unique(labels)
    # TODO : matrice n x k des distances moyennes à chaque cluster ; corriger a(p) pour exclure p
    raise NotImplementedError("À compléter")

In [ ]:
# Test
X, y = make_blobs(n_samples=400, centers=4, random_state=2)
lab = KMeans(4, n_init=5, random_state=0).fit_predict(X)
check_close(silhouette(X, lab), silhouette_samples(X, lab), "silhouette = sklearn", rtol=1e-6, atol=1e-8)

In [ ]:
X, y = make_blobs(n_samples=1200, centers=4, cluster_std=1.6, random_state=11)
ks = range(2, 11); costs, sils = [], []
for k in ks:
    best = min((lloyd(X, init_kmeanspp(X, k, np.random.default_rng(s))) for s in range(5)),
               key=lambda r: r[2][-1])
    costs.append(best[2][-1]); sils.append(silhouette(X, best[1]).mean())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(*X.T, s=5, color=INK2); axes[0].set_title("Données (4 groupes générés)")
axes[1].plot(list(ks), costs, marker="o"); axes[1].set_title("Coude : variance vs k"); axes[1].set_xlabel("k")
axes[2].plot(list(ks), sils, marker="o", color=ORANGE); axes[2].set_title("Silhouette moyenne vs k"); axes[2].set_xlabel("k")
axes[2].axvline(list(ks)[int(np.argmax(sils))], color=GRAY, ls="--", lw=1)
plt.tight_layout(); plt.show()
print("k choisi par la silhouette :", list(ks)[int(np.argmax(sils))])

**Lecture.** Deux des quatre groupes générés se chevauchent : le coude suggère $k = 4$ alors que la silhouette préfère $k = 3$ (elle pénalise des clusters voisins mal séparés). Les critères peuvent diverger : le choix de $k$ reste un **choix de modélisation**, à croiser avec l'usage prévu.

## Partie 6 — Application : regrouper des actions par comportement

Question classique en gestion quantitative : les **secteurs officiels** (GICS) décrivent-ils bien la structure de co-mouvements des actions ? Si l'on regroupe les actions par la ressemblance de leurs rendements, retrouve-t-on les secteurs ?

**Représentation.** Chaque action est un point de $\mathbb{R}^T$ : son vecteur de rendements quotidiens **standardisés** $z_i$. Pour deux tels vecteurs, $\|z_i - z_j\|^2 = 2T(1 - \rho_{ij})$ : la distance euclidienne est une fonction décroissante de la **corrélation**. k-means regroupe donc les actions fortement corrélées.

**Piège.** Toutes les actions partagent un fort facteur **marché** qui domine les corrélations. On compare donc deux versions : rendements bruts, et rendements **neutralisés du marché** (on retire, par régression, le rendement moyen de l'univers).

### Exercice 6.1

In [ ]:
rets, sectors, src = tp_data.load_stock_returns()
print("source :", src, "|", rets.shape[1], "actions,", rets.shape[0], "jours,", sectors.nunique(), "secteurs")

def standardize_cols(R):
    return ((R - R.mean()) / R.std()).to_numpy().T          # (n_actions, T)

def market_neutral(R):
    """Résidus de la régression de chaque action sur le rendement moyen de l'univers."""
    # TODO : beta_i = cov(r_i, m) / var(m) ; résidu = r_i - beta_i m
    raise NotImplementedError("À compléter")

def best_kmeans(X, k, n_init=10, seed=0):
    """Lloyd + k-means++, meilleur de n_init redémarrages. Retourne (centres, étiquettes, coût)."""
    # TODO : n_init appels à lloyd(X, init_kmeanspp(...)) avec des graines différentes ; garder le coût final minimal
    raise NotImplementedError("À compléter")

k = sectors.nunique()
_, lab_raw, _ = best_kmeans(standardize_cols(rets), k)
_, lab_mn, _ = best_kmeans(standardize_cols(market_neutral(rets)), k)
ari_raw = adjusted_rand_score(sectors, lab_raw); ari_mn = adjusted_rand_score(sectors, lab_mn)
observe(ari_mn > ari_raw - 0.05, "neutraliser le marché ne dégrade pas l'accord avec les secteurs")
print(f"ARI clusters vs secteurs — rendements bruts : {ari_raw:.2f} | neutralisés du marché : {ari_mn:.2f}")

In [ ]:
pd.crosstab(sectors.to_numpy(), lab_mn, rownames=["secteur GICS"], colnames=["cluster"])

**Questions.**
1. L'**ARI** (*adjusted Rand index*) vaut 1 pour un accord parfait et ≈ 0 pour un clustering aléatoire. Comment interprétez-vous les deux valeurs ?
2. Quels secteurs sont les mieux / les moins bien identifiés ? Pourquoi ?
3. Pourquoi k-means est-il un choix discutable pour ce problème, et que proposera le TP 3 ?

> ✍️ *Votre réponse :* …

## Partie 7 — Synthèse : quel algorithme, quand ?

| Méthode | Coût | Optimalité | Quand l'utiliser |
|---|---|---|---|
| Centroïde ($k=1$) | $O(nd)$ | exacte | résumé d'un nuage ; brique de Lloyd |
| **DP 1D** | $O(kn^2)$ | **exacte** | une seule variable (seuils de régimes, *bucketing* d'un signal, discrétisation) |
| Énumération des partitions de Voronoï | $O(n^{kd+1})$ | exacte | jamais en pratique (NP-difficile en général) |
| **Lloyd** | $O(nkd)$ par itération | minimum local | cas général ; rapide, simple |
| + init. uniforme / Forgy / partition aléatoire | $O(kd)$ à $O(nd)$ | aucune garantie | à éviter dès que $k$ est grand |
| + **k-means++** | $O(nkd)$ | $O(\log k)$-approx. en espérance | **défaut** ; combiner avec plusieurs redémarrages |

**Choix de $k$ :** le **coude** est visuel et ambigu ; la **silhouette** donne un critère chiffré (coût $O(n^2)$ : sous-échantillonner si $n$ est grand). Toujours croiser avec le sens métier.

**Limites de k-means (et alternatives vues dans le cours)**
- clusters supposés convexes, isotropes, de tailles voisines → **GMM** (TP 4) pour des clusters elliptiques et une affectation probabiliste ;
- besoin de $k$ et sensibilité à l'échelle des variables → standardiser ; **clustering hiérarchique** (TP 3) pour une vue multi-échelle ;
- sensibilité aux outliers (moyenne) → k-médoïdes, ou nettoyage préalable.